# ClusterBP v2: more baselines — LightGBM, CatBoost and MiniRocket

**Tabular baselines.** LightGBM and CatBoost on the same windows, folds and features as XGB_full
(`FEATURE_SETS['full']`), with early stopping on AUPRC and the Macro F1 threshold on the fold's inner validation.
Rows are added to `results_v2/additional/tabular_models.csv` in the same format as Random Forest and
HistGradientBoosting: AUROC, AUPRC with 95% CI, ΔAUPRC against XGB_full with 95% CI (patient-level bootstrap).

**MiniRocket** (Dempster, Schmidt & Webb, KDD 2021) on daily means of SBP, DBP and HR over the 14 days of the
history window (3 channels × 14 days; days without measurements are filled forward, then backward). Transform:
`MiniRocket` in `clusterbp_extra.py` (numpy reimplementation of the reference code; 1,008 features, since a 14-point
series allows only dilation 1). Classifier: ridge regression on standardised features with α chosen by AUPRC on the
inner validation and Platt scaling on the same set. Evaluated alone and stacked with XGB_full (logistic regression
on the two logits, trained on inner validation), as the other sequence models. Rows are added to
`results_v2/sequence_models.csv` with differences against GRU and against GRU + XGB_full stacking.

Rows with the same target and model are replaced; other rows are kept. Before writing, the current `results_v2/`
is copied once to `results_v2_before_eda_baselines/`.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from clusterbp_lib import (DATA_DIR, RESULTS_DIR, SEED, N_FOLDS, FEATURE_SETS, load_cached, fold_masks,
                           add_phenotype_columns, load_gru, load_xgb, gru_logits, sigmoid, logit, best_threshold,
                           boot_eval, ci)
from clusterbp_extra import snapshot_results, upsert_rows, daily_series, MiniRocket, fit_ridge, N_BOOT

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())
snapshot_results()
TAB_TARGETS = ['A', 'B', 'OLD']      # as in additional/tabular_models.csv
SEQ_TARGETS = ['A', 'B']             # as in sequence_models.csv
FULL = FEATURE_SETS['full']
print('LightGBM', lgb.__version__)

win, X_seq, seq_len, pheno_win, oof = load_cached()
pid = win['pid'].values
print(f'Windows: {len(win):,}, patients: {len(np.unique(pid)):,}, features: {len(FULL)}')

Copy results_v2_before_eda_baselines/ already exists and is kept as is
LightGBM 4.6.0


Windows: 420,355, patients: 9,413, features: 95


## 1. LightGBM and CatBoost

Settings follow XGB_full (`fit_xgb` in `clusterbp_lib.py`): up to 2,000 trees, learning rate 0.05, early stopping
after 100 rounds without improvement of AUPRC on the inner validation; trees of comparable size (LightGBM: 31 leaves;
CatBoost: symmetric trees of depth 6), row and column subsampling 0.8 where available.

In [2]:
def fit_lgb(Xtr, ytr, Xva, yva, seed):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, metric='average_precision', n_jobs=-1, random_state=seed,
                           verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(100, verbose=False)])
    return m, m.best_iteration_


def fit_cat(Xtr, ytr, Xva, yva, seed):
    m = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6, rsm=0.8, eval_metric='PRAUC',
                           od_type='Iter', od_wait=100, use_best_model=True, random_seed=seed, thread_count=-1,
                           verbose=False, allow_writing_files=False)
    m.fit(Xtr, ytr, eval_set=(Xva, yva))
    return m, m.get_best_iteration()


TAB = {'LightGBM': fit_lgb, 'CatBoost': fit_cat}
tab = {t: {} for t in TAB_TARGETS}
tab_info = []
t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    add_phenotype_columns(win, pheno_win, k)
    X = win[FULL].values.astype(np.float32)
    for t in TAB_TARGETS:
        y = win[f'y_{t}'].values
        for name, fit in TAB.items():
            t0 = time.time()
            m, best_it = fit(X[tr], y[tr], X[va], y[va], SEED + k)
            s = m.predict_proba(X)[:, 1]
            thr, _ = best_threshold(y[va], s[va])
            st = tab[t].setdefault(name, [np.full(len(win), np.nan), np.zeros(len(win), np.int8)])
            st[0][te] = s[te]
            st[1][te] = s[te] > thr
            tab_info.append({'target': t, 'fold': k, 'model': name, 'best iteration': best_it,
                             'seconds': time.time() - t0})
    print(f'Fold {k}: {(time.time() - t_all) / 60:.1f} min')


def paired(name, y, models, labels, ref):
    """Metrics and bootstrap differences relative to model ref (same as in clusterbp_v2_additional.ipynb)."""
    B = boot_eval(y.astype(float), pid, models, n_boot=N_BOOT)
    rows = []
    for j, lab in enumerate(labels):
        s, p = models[j]
        d = B[:, j, 1] - B[:, labels.index(ref), 1]
        rows.append({'target': name, 'model': lab, 'AUROC': roc_auc_score(y, s), 'AUPRC': average_precision_score(y, s),
                     'AUPRC 95% CI': ci(B[:, j, 1]), 'macro_F1': f1_score(y, p, average='macro'),
                     f'ΔAUPRC vs {ref}': average_precision_score(y, s) - average_precision_score(y, models[labels.index(ref)][0]),
                     'ΔAUPRC 95% CI': ci(d) if lab != ref else '',
                     'p': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())) if lab != ref else np.nan})
    return pd.DataFrame(rows)


frames = []
for t in TAB_TARGETS:
    y = win[f'y_{t}'].values
    labels = ['XGB_full'] + list(TAB)
    models = [(oof[f'{t}__XGB_full__score'].values, oof[f'{t}__XGB_full__pred'].values.astype(float))]
    models += [(tab[t][m][0], tab[t][m][1].astype(float)) for m in TAB]
    frames.append(paired(t, y, models, labels, 'XGB_full'))
new_tab = pd.concat(frames)
new_tab = new_tab[new_tab['model'] != 'XGB_full']          # the XGB_full rows are already in the table
tabular = upsert_rows(os.path.join(RESULTS_DIR, 'additional', 'tabular_models.csv'), new_tab)
pd.DataFrame(tab_info).to_csv(os.path.join(RESULTS_DIR, 'additional', 'tabular_models_training.csv'), index=False)
display(tabular.round(4))
display(pd.DataFrame(tab_info).groupby(['target', 'model'])[['best iteration', 'seconds']].mean().round(1))

Fold 0: 2.9 min


Fold 1: 6.3 min


Fold 2: 8.0 min


Fold 3: 9.6 min


Fold 4: 11.3 min


,target,model,AUROC,AUPRC,AUPRC 95% CI,macro_F1,ΔAUPRC vs XGB_full,ΔAUPRC 95% CI,p
0,A,XGB_full,0.9338,0.5423,[0.514; 0.568],0.7531,0.0000,NaN,NaN
1,A,LR,0.9272,0.5005,[0.470; 0.529],0.7392,-0.0417,[-0.047; -0.036],0.000
2,A,Random Forest,0.9306,0.5371,[0.508; 0.563],0.7502,-0.0051,[-0.007; -0.003],0.000
3,A,HistGradientBoosting,0.9307,0.5243,[0.497; 0.549],0.7437,-0.0180,[-0.021; -0.015],0.000
12,A,LightGBM,0.9327,0.5394,[0.511; 0.565],0.7508,-0.0029,[-0.005; -0.001],0.002
13,A,CatBoost,0.9335,0.5411,[0.514; 0.566],0.7529,-0.0012,[-0.003; 0.001],0.246
4,B,XGB_full,0.9649,0.8518,[0.843; 0.860],0.8622,0.0000,NaN,NaN
5,B,LR,0.9524,0.8015,[0.791; 0.812],0.8457,-0.0503,[-0.054; -0.047],0.000
6,B,Random Forest,0.9636,0.8482,[0.839; 0.857],0.8599,-0.0036,[-0.004; -0.003],0.000
7,B,HistGradientBoosting,0.9641,0.8485,[0.840; 0.857],0.8603,-0.0033,[-0.004; -0.002],0.000


best iteration  seconds
target model                            
A      CatBoost           288.8     27.2
       LightGBM            89.0      2.9
B      CatBoost           720.0     66.2
       LightGBM           230.8      6.6
OLD    CatBoost           285.2     26.8
       LightGBM           164.2      4.9

## 2. MiniRocket on daily means

The daily series are built once and cached (`cache_v2/daily14.npy`). The MiniRocket transform is fitted on the
training windows of each fold (biases come from random training examples), so test windows never influence it.

In [3]:
path = os.path.join(DATA_DIR, 'daily14.npy')
if os.path.exists(path) and np.load(path, mmap_mode='r').shape[0] == len(win):
    D = np.load(path)
else:
    t0 = time.time()
    meas = pd.read_parquet(os.path.join(DATA_DIR, 'measurements.parquet'))
    D = daily_series(win, meas)
    np.save(path, D)
    print(f'Daily series built in {time.time() - t0:.0f} s')
print('Daily series:', D.shape, '; share of days without measurements (filled):',
      round(float(1 - np.mean(np.isfinite(D))), 4))
D = np.nan_to_num(D, nan=float(np.nanmean(D)))

store = {t: {} for t in SEQ_TARGETS}
mr_info = []


def put(t, name, te, s, p):
    st = store[t].setdefault(name, [np.full(len(win), np.nan), np.zeros(len(win), np.int8)])
    st[0][te] = s[te]
    st[1][te] = p[te]


t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    add_phenotype_columns(win, pheno_win, k)
    t0 = time.time()
    mr = MiniRocket(seed=SEED + k).fit(D[tr])
    F = mr.transform(D)
    t_transform = time.time() - t0
    for t in SEQ_TARGETS:
        y = win[f'y_{t}'].values
        px = load_xgb('full', t, k).predict_proba(win[FULL].values)[:, 1]
        gm, mu, sd, corr = load_gru(t, k)
        probs = {'GRU': sigmoid(gru_logits(gm, X_seq, seq_len, mu, sd) + corr)}
        t0 = time.time()
        probs['MiniRocket'], alpha = fit_ridge(F, y, tr, va)
        mr_info.append({'target': t, 'fold': k, 'model': 'MiniRocket', 'features': F.shape[1], 'ridge alpha': alpha,
                        'seconds': t_transform + time.time() - t0})
        for name, ps in probs.items():
            thr, _ = best_threshold(y[va], ps[va])
            put(t, name, te, ps, (ps > thr).astype(np.int8))
            st = LogisticRegression(max_iter=3000).fit(np.column_stack([logit(ps[va]), logit(px[va])]), y[va])
            s = st.predict_proba(np.column_stack([logit(ps), logit(px)]))[:, 1]
            thr, _ = best_threshold(y[va], s[va])
            put(t, f'{name} + XGB_full (stacking)', te, s, (s > thr).astype(np.int8))
    del F
    print(f'Fold {k}: {(time.time() - t_all) / 60:.1f} min')
pd.DataFrame(mr_info).to_csv(os.path.join(RESULTS_DIR, 'minirocket_training.csv'), index=False)

Daily series built in 4 s
Daily series: (420355, 3, 14) ; share of days without measurements (filled): 0.0


Fold 0: 2.3 min


Fold 1: 4.6 min


Fold 2: 6.8 min


Fold 3: 9.0 min


Fold 4: 11.2 min


In [4]:
rows = []
for t in SEQ_TARGETS:
    y = win[f'y_{t}'].values.astype(float)
    labels = list(store[t])
    models = [(store[t][n][0], store[t][n][1].astype(float)) for n in labels]
    B = boot_eval(y, pid, models, n_boot=N_BOOT)
    for j, lab in enumerate(labels):
        if not lab.startswith('MiniRocket'):
            continue                                  # GRU rows are already in sequence_models.csv
        ref = 'GRU + XGB_full (stacking)' if 'stacking' in lab else 'GRU'
        jr = labels.index(ref)
        d = B[:, j, 1] - B[:, jr, 1]
        rows.append({'target': t, 'model': lab, 'AUROC': roc_auc_score(y, models[j][0]),
                     'AUPRC': average_precision_score(y, models[j][0]), 'AUPRC 95% CI': ci(B[:, j, 1]),
                     'macro_F1': f1_score(y, models[j][1], average='macro'),
                     'ΔAUPRC vs ' + ref: f'{np.mean(d):+.4f} {ci(d)}'})
seq = upsert_rows(os.path.join(RESULTS_DIR, 'sequence_models.csv'), pd.DataFrame(rows))
pd.set_option('display.max_colwidth', 60)
display(seq.round(4))
display(pd.DataFrame(mr_info).groupby('target')[['ridge alpha', 'seconds']].median())

,target,model,AUROC,AUPRC,AUPRC 95% CI,macro_F1,ΔAUPRC vs GRU,ΔAUPRC vs GRU + XGB_full (stacking)
0,A,GRU,0.9164,0.4748,[0.446; 0.501],0.7304,NaN,NaN
1,A,GRU + XGB_full (stacking),0.9345,0.5446,[0.517; 0.570],0.7537,NaN,NaN
2,A,TimeAwareGRU,0.9170,0.4684,[0.438; 0.495],0.7304,-0.0063 [-0.010; -0.003],NaN
3,A,TimeAwareGRU + XGB_full (stacking),0.9346,0.5444,[0.516; 0.570],0.7536,NaN,-0.0002 [-0.001; 0.000]
4,A,TCN,0.9163,0.4690,[0.440; 0.496],0.7311,-0.0058 [-0.010; -0.001],NaN
5,A,TCN + XGB_full (stacking),0.9346,0.5439,[0.516; 0.569],0.7535,NaN,-0.0007 [-0.001; -0.000]
6,A,Transformer,0.9184,0.4743,[0.445; 0.501],0.7325,-0.0006 [-0.006; 0.005],NaN
7,A,Transformer + XGB_full (stacking),0.9347,0.5444,[0.516; 0.570],0.7536,NaN,-0.0003 [-0.001; 0.000]
16,A,MiniRocket,0.8830,0.3468,[0.316; 0.375],0.6837,-0.1279 [-0.138; -0.118],NaN
17,A,MiniRocket + XGB_full (stacking),0.9339,0.5424,[0.514; 0.568],0.7531,NaN,-0.0022 [-0.003; -0.001]


,ridge alpha,seconds
target,,
A,100000.00000,84.285252
B,3162.27766,82.070791
